# AWS (amazon web service) Rekognition

## AWS

AWS は世界で最もよく使われるクラウドサービスである。

Rekognition は、AWSの中でAIを使った画像認識 (Image Recognition) サービスである。
API (Application Program Interface) を使って使用する。

---


## Using Rekognition API


### Loading library required
```
import requests
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib import patches
from io import BytesIO
from IPython.core.display import HTML
import base64
import numpy as np
import requests
%matplotlib inline
```

In [ ]:
import requests
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib import patches
from io import BytesIO
from IPython.core.display import HTML
import base64
import numpy as np
import requests
%matplotlib inline

AWS API 通信用ライブラリ boto3 をインストール
```
!pip install boto3
```


In [ ]:
!pip install boto3

boto3 と OS ファイル操作ライブラリ sys を読み込む
```
import boto3
import sys
```

In [ ]:
import boto3
import sys

### AWS API access key
API使用のためのアクセスキーの設定をして、通信クライアントオブジェクト client を生成・初期化

```
client = boto3.client('rekognition','us-west-2',
                      aws_access_key_id="",
                      aws_secret_access_key="",
                      aws_session_token=""
                      )

```

キーの文字列は授業中に指示を受けること

### Google Drive Mount
Google Drive を Colab から読み書きできるようにする（mount）

```
from google.colab import drive
drive.mount('/content/drive')
```
自分のアカウントの Google Drive が /content/drive に見えるようになる。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

確認
```
!ls /content/drive

!ls /content/drive/MyDrive
```

### Google Drive に画像を置く

授業用 Google Drive フォルダの
「第2回AI-Hands-on-0417」
から
「professor.jpg」
をダウンロードし、自分の colab でマウントした Google Drive のマイドライブに置く。

```
!ls /content/drive/MyDrive
```
で確認

In [ ]:
# Google Drive の MyDrive に置いた画像を認識したい
# source_bytes に画像のファイルを読み込む

with open("/content/drive/MyDrive/lab2.png", "rb") as source_image:
  source_bytes = source_image.read()

In [ ]:
# URL で指定された画像を認識したい
# source_bytes に画像のファイルを読み込む

#source_bytes = requests.get("").content

### Call Rekognition API

client オブジェクトに読み込んだ source_bytes を渡して、AWS に送信し、認識結果を response オブジェクトで受け取り代入。

```
response = client.detect_faces(Image={ 'Bytes': source_bytes},Attributes=['ALL'])
```

In [ ]:
response = client.detect_faces(Image={ 'Bytes': source_bytes},Attributes=['ALL'])

試しに認識した年齢を見てみる

```
ageRange = response['FaceDetails'][0]['AgeRange']
print('年齢：' + str(ageRange['Low']) + ' - ' + str(ageRange['High']))
```

In [ ]:
ageRange = response['FaceDetails'][0]['AgeRange']
print('年齢：' + str(ageRange['Low']) + ' - ' + str(ageRange['High']))

response にどんな情報があるか見てみる
```
print(response)
```

In [ ]:
print(response)

response オブジェクトの型を確認する
```
type(response)
```

In [ ]:
type(response)

response オブジェクトの長さを確認

```
len(response)
```

In [ ]:
len(response)

### オブジェクトを見やすくする

```
from pprint import pprint
```

In [ ]:
from pprint import pprint

In [ ]:
pprint(response)

```
pprint(response['FaceDetails'][0], indent=2)
```

In [ ]:
pprint(response['FaceDetails'][0], indent=2)

### 返ってきた response オブジェクトを更に確認

```
len(response['FaceDetails'])
```

In [ ]:
len(response['FaceDetails'])

### 顔認識結果を可視化する

```
faces = response['FaceDetails']
```

In [ ]:
faces = response['FaceDetails']

In [ ]:
image = Image.open(BytesIO(source_bytes))
image.size

In [ ]:
image = Image.open(BytesIO(source_bytes))
plt.figure(figsize=(8, 8))
ax = plt.imshow(image)
for face in faces:
  # 描きたい四角の左上端の位置（画素x座標）
  fleft = face['BoundingBox']['Left']    * image.size[0]
  # 描きたい四角の左上端の位置（画素y座標）
  ftop = face['BoundingBox']['Top']      * image.size[1]
  # 描きたい四角の横幅画素数
  fwidth = face['BoundingBox']['Width']  * image.size[0]
  # 描きたい四角の縦高さ画素数
  fheight = face['BoundingBox']['Height'] * image.size[1]

  # 年齢のレンジ（最低～最高）
  fage = face['AgeRange']
  # 性別
  fgen = face['Gender']
  # 上記の情報で各顔の位置に青（男性）または赤（女性）の四角を描く
  p = patches.Rectangle(
      (fleft, ftop), fwidth, fheight,
      fill=False, linewidth=2,
      color='blue' if fgen['Value']=='Male' else 'red') # 三項演算子（条件付き代入）
  ax.axes.add_patch(p)
  plt.text(fleft, ftop,"%s, %s"%( ((fage['High']+fage['Low'])/2) ,fgen['Value']), color='white')
_ = plt.axis("off")

## Let's try using other pictures.

Upload your own picture using Jupyter "Upload" button on "Home" directory.


### カメラアプリ
JavaScript でPC接続カメラを Open し、img に保存する。
このカメラアプリから撮影した画像を AWK Rekognition で認識してみる。

In [ ]:
def camera_on():
  from IPython.display import HTML, Image
  from google.colab.output import eval_js
  from base64 import b64decode
  from io import BytesIO
  from PIL import Image

  canvas_html = """
  <style type="text/css">
  #video { border: 1px solid yellow; width: 640px; height: 480px;}
  </style>

  <button>保存！</button>
  <video id="video"></video>
  <canvas width=640 height=480></canvas>

  <script>
  var canvas = document.querySelector('canvas')
  var button = document.querySelector('button')
  const video = document.querySelector('#video');

  navigator.mediaDevices.getUserMedia({ video: true, audio: false })
          .then((stream) => {
              video.srcObject = stream;
              video.play();
          })

  var data = new Promise(resolve=>{
    button.onclick = ()=>{
      const context = canvas.getContext("2d");
      context.drawImage(video, 0, 0, canvas.width, canvas.height);
      resolve(canvas.toDataURL('image/png'))
    }
  })
  </script>
  """

  # HTML表示
  display(HTML(canvas_html))
  data = eval_js("data")
  # ボタンが押された後，canvasデータがbinaryに入る (BASE64エンコードで来るのでデコード)
  binary = b64decode(data.split(',')[1])
  img = Image.open(BytesIO(binary))
  # 背景が透明なため，白背景画像 background を作り書いた文字を重ね，img に入れる
  background = Image.new("RGB", img.size, (255, 255, 255))
  background.paste(img, img)
  img = background
  return img

## 注意！！
- 最初，ブラウザからカメラ起動の許可が聞かれたら許可する．
- 「VMWare virtual」等と表示されているカメラは選択しない．
- 1回目は映像取得に失敗するかも知れないので，停止ボタンを押し，再度セル実行する．
- 黄色の枠内に映像が表示されれば良い．

In [ ]:
img = camera_on()

In [ ]:
display(img)

In [ ]:
img_bytes = BytesIO()
img.save(img_bytes, format='PNG')
response = client.detect_faces(Image={ 'Bytes': img_bytes.getvalue()},Attributes=['ALL'])